## Importaciones de Librerías

In [ ]:
import pandas as pd

from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score

import lightgbm as lgb

import optuna

import joblib

import requests

from dotenv import load_dotenv
import os

## Carga de Datos

In [ ]:
load_dotenv()
URL_BACKEND_INTERNSHIPS = os.getenv('URL_BACKEND_INTERNSHIPS')

In [ ]:
def get_internships():
    try:
        response = requests.get(URL_BACKEND_INTERNSHIPS)
        if response.status_code == 200:
            return response.json()
        else:
            print(f'Error al obtener los datos: {response.status_code} - {response.text}')
            return None
    except Exception as e:
        print(f'Error con la conexión a la API de las prácticas: {e}')
        return None

In [ ]:
df = pd.DataFrame(get_internships())
print("Datos cargados exitosamente")

## Preparación Varibales x, y

In [ ]:
# Definimos solo las variables que demostraron verdadero valor predictivo
variables_seleccionadas = [
    'interview_score', 'skills_score', 'communication_score', 'coding_test_score',
    'projects_count', 'soft_skills_score', 'certifications_count', 'resume_score',
    'internships_done', 'extracurricular', 'consistency_score', 'placement_training',
    'cgpa', 'github_score'
]

X = df[variables_seleccionadas]
y = df['selected']

## Separación varibales en entreno y test

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, train_size=0.8, random_state=42, stratify=y)

## Búsuqueda de Mejores Hiperparametros con Optuna

In [ ]:
def objective_lightgbm(trial):
    params = {
        'boosting': trial.suggest_str('boosting', 'rf', 'gbdt'),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.2),
        'max_leaves': trial.suggest_int('max_leaves', 10, 35),
        'max_depth': trial.suggest_int('max_depth', 5, 20),
        'objective': 'binary',
        'random_state': 42
    }

    model = lgb(**params)
    return cross_val_score(model, X_train, y_train, cv=5, scoring='roc_auc').mean()
study_lgb = optuna.create_study(direction='maximize')
study_lgb.optimize(objective_lightgbm, n_trials=25)
print('LightGBM - Mejores Hiperparámetros: ', study_lgb.best_params)

## Pipeline

In [ ]:
pipeline = Pipeline(
    steps = [
        ('model', lgb(**study_lgb.best_params, eval_metric='logloss', random_state=42))
    ]
)

In [ ]:
print("Entrenando el modelo final con los mejores parámetros...")
pipeline.fit(X_train, y_train)

In [ ]:
# Calcular predicciones para ambos conjuntos
y_train_pred_proba = pipeline.predict_proba(X_train)[:, 1]
y_test_pred_proba = pipeline.predict_proba(X_test)[:, 1]

# Calcular el ROC-AUC para ambos
auc_train = roc_auc_score(y_train, y_train_pred_proba)
auc_test = roc_auc_score(y_test, y_test_pred_proba)

print(f"ROC-AUC Entrenamiento: {auc_train:.4f}")
print(f"ROC-AUC Prueba (Test):  {auc_test:.4f}")

# Calcular la brecha (gap)
brecha = auc_train - auc_test
print(f"Brecha de Overfitting:   {brecha:.4f}")

In [ ]:
print("\nEvaluando el modelo en el conjunto de prueba (X_test)...")
y_pred = pipeline.predict(X_test)
y_pred_proba = pipeline.predict_proba(X_test)[:, 1]

In [ ]:
print("\n=== Reporte de Clasificación ===")
print(classification_report(y_test, y_pred))

print("=== Matriz de Confusión ===")
print(confusion_matrix(y_test, y_pred))

print(f"\nROC-AUC Score en Test: {roc_auc_score(y_test, y_pred_proba):.4f}")

## Guardado del modelo

In [ ]:
joblib.dump(pipeline, '../models/lgb_model.pkl')